In [1]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---


# 🏆 V6: THE UNBEATABLE SOLUTION — PSTU DataThon 2026

**Target LB: 0.28–0.32 | Pure LogLoss | Zero SMOTE | Adversarial Validation | PCA(ddof=0) | Pseudo-Labeling**

---

## 🔴 Why V5 Failed (OOF F1 = 0.067 — WORST VERSION EVER)

| Model | V3 OOF | V4 OOF | V5 OOF | Root Cause |
|-------|--------|--------|--------|------------|
| XGBoost | 0.346 | ~0.310 | **0.107** | SMOTE+weight double-correction |
| LightGBM | 0.000 | ~0.330 | **0.003** | SMOTE killed it AGAIN |
| CatBoost | 0.154 | 0.000 | **0.003** | SMOTE+weight = guaranteed death |

**THE PATTERN IS CLEAR**: SMOTE + ANY class_weight = MODEL DEATH across all versions.
V5 proved that even with only 6 true cats, SMOTE kills CB+LGB and cripples XGB to 1/3 strength.

---

## ✅ V6: COMPLETELY DIFFERENT PARADIGM

### What We REMOVE:
- ❌ **NO SMOTE** — the common failure across V1-V5
- ❌ **NO scale_pos_weight** — no class weights of any kind
- ❌ **NO auto_class_weights** — CatBoost learns naturally
- ❌ **NO threshold=0.5 assumption** — we FIND the optimal threshold

### What We ADD:
- ✅ **Adversarial Validation** — detect & drop train/test distribution-shifted features
- ✅ **PCA with ddof=0** — mathematically correct variance, latent representations
- ✅ **Pure LogLoss** — models learn raw, well-calibrated probabilities
- ✅ **Optimal Threshold Search** — grid search 0.01→0.50 on OOF for max F1
- ✅ **Pseudo-Labeling** — confident test predictions → augment training → retrain
- ✅ **Binary Submission** — probabilities converted to 0/1 using optimal threshold

---

## 📋 V6 Strategy Pipeline

```
STEP 1: Adversarial Validation → Drop covariate-shifted features
STEP 2: QT + PCA(ddof=0) + 6 Label-Encoded Cats → ~120 clean features
STEP 3: XGB+LGB+CB × 3 seeds × 5 folds → Pure LogLoss, no weights
STEP 4: Threshold grid search on OOF → Find optimal F1 cutoff
STEP 5: Pseudo-label confident test samples → Augment train → Retrain ALL
STEP 6: Apply optimal threshold → Binary 0/1 submission
```


# CELL 1: Imports & Environment Setup


In [2]:
# ===================================================================
# CELL 1: Imports & Environment Setup
# ===================================================================
import numpy as np
import pandas as pd
import warnings, os, gc, sys, time
from pathlib import Path
warnings.filterwarnings('ignore')

# Core ML
from sklearn.preprocessing import QuantileTransformer, LabelEncoder, RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.neighbors import NearestNeighbors

# Models
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

print("=" * 70)
print("  V6: THE UNBEATABLE SOLUTION")
print("  Pure LogLoss | Zero SMOTE | Adversarial Validation | PCA(ddof=0)")
print("  Pseudo-Labeling | Optimal Threshold | Binary Submission")
print("=" * 70)
print(f"  Python: {sys.version.split()[0]}")
print(f"  NumPy:  {np.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Start:  {time.strftime('%Y-%m-%d %H:%M:%S')}")

T_START = time.time()

# ===================================================================
# Debug Helper
# ===================================================================
class DebugTracker:
    def __init__(self):
        self.log = []
        self.step_count = 0
        self.t_last = time.time()

    def log_step(self, name, shape=None, extra=None):
        self.step_count += 1
        t_now = time.time()
        elapsed = t_now - self.t_last
        self.t_last = t_now
        shape_str = f"  shape={shape}" if shape else ""
        time_str = f"  +{elapsed:.1f}s"
        extra_str = f"  {extra}" if extra else ""
        print(f"  [D{self.step_count:02d}] {name}{shape_str}{time_str}{extra_str}")
        self.log.append({'step': self.step_count, 'name': name, 'shape': shape, 'elapsed': elapsed, 'extra': extra})

    def summary(self):
        print("\n" + "=" * 70)
        print("  DEBUG TRACKER SUMMARY")
        print("=" * 70)
        for e in self.log:
            print(f"  [{e['step']:02d}] {e['name']}: shape={e.get('shape','N/A')}")

dbg = DebugTracker()


  V6: THE UNBEATABLE SOLUTION
  Pure LogLoss | Zero SMOTE | Adversarial Validation | PCA(ddof=0)
  Pseudo-Labeling | Optimal Threshold | Binary Submission
  Python: 3.12.13
  NumPy:  2.0.2
  Pandas: 2.3.3
  Start:  2026-08-11 06:52:08


# CELL 2: Configuration


In [3]:
# ===================================================================
# CELL 2: Configuration
# ===================================================================
CFG = {
    # === Paths (Kaggle) ===
    'train_path': '/kaggle/input/competitions/pstu-data-thon-2026-vol-1/train.csv',
    'test_path':  '/kaggle/input/competitions/pstu-data-thon-2026-vol-1/test.csv',
    'sub_path':   '/kaggle/input/competitions/pstu-data-thon-2026-vol-1/sample_submission.csv',

    # === Reproducibility ===
    'seed': 42,
    'ensemble_seeds': [42, 123, 456],

    # === CV ===
    'n_folds': 5,

    # === Adversarial Validation ===
    'adv_drop_top_pct': 5.0,    # Drop top 5% most drift-heavy features
    'adv_n_estimators': 200,    # Quick LGB for drift detection

    # === PCA (ddof=0) ===
    'pca_variance_threshold': 0.95,

    # === Feature Engineering ===
    'use_row_stats': True,
    'use_kmeans': True,
    'kmeans_clusters': [8, 16],

    # === Model Parameters (NO class weights, NO SMOTE) ===

    # XGBoost — pure LogLoss
    'xgb_params': {
        'n_estimators': 3000,
        'max_depth': 6,
        'learning_rate': 0.015,
        'subsample': 0.80,
        'colsample_bytree': 0.75,
        'colsample_bylevel': 0.70,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 2.0,
        'min_child_weight': 5,
        'tree_method': 'hist',
        'eval_metric': 'logloss',
        'early_stopping_rounds': 200,
        'verbosity': 0,
        'random_state': 42,
        # DELIBERATELY OMITTED: scale_pos_weight
    },

    # LightGBM — pure LogLoss
    'lgb_params': {
        'n_estimators': 3000,
        'max_depth': 6,
        'learning_rate': 0.015,
        'subsample': 0.80,
        'colsample_bytree': 0.75,
        'reg_alpha': 0.1,
        'reg_lambda': 2.0,
        'num_leaves': 63,
        'min_child_samples': 30,
        'random_state': 42,
        'verbosity': -1,
        # DELIBERATELY OMITTED: scale_pos_weight
    },

    # CatBoost — pure LogLoss
    'catboost_params': {
        'n_estimators': 3000,
        'max_depth': 6,
        'learning_rate': 0.020,
        'l2_leaf_reg': 5.0,
        'random_strength': 1.0,
        'bagging_temperature': 0.5,
        'border_count': 254,
        'min_data_in_leaf': 30,
        'random_seed': 42,
        'verbose': 0,
        'allow_writing_files': False,
        'early_stopping_rounds': 200,
        # DELIBERATELY OMITTED: auto_class_weights, scale_pos_weight
    },

    # === Threshold Search ===
    'threshold_min': 0.01,
    'threshold_max': 0.50,
    'threshold_step': 0.005,

    # === Pseudo-Labeling ===
    'pseudo_pos_percentile': 99.0,   # Top 1% most confident → class 1
    'pseudo_neg_percentile': 30.0,   # Bottom 30% most confident → class 0
    'pseudo_min_samples': 50,        # Minimum pseudo-positive samples required
}

# Print configuration
print("\n" + "=" * 70)
print("  CONFIGURATION (CFG)")
print("=" * 70)
print(f"  Ensemble seeds:     {CFG['ensemble_seeds']}")
print(f"  N folds:            {CFG['n_folds']}")
print(f"  Stage 1 models:     {len(CFG['ensemble_seeds']) * CFG['n_folds'] * 3}")
print(f"  Total (both stages): {len(CFG['ensemble_seeds']) * CFG['n_folds'] * 3 * 2}")
print(f"  Adv drop top:       {CFG['adv_drop_top_pct']}% features")
print(f"  PCA variance keep:  {CFG['pca_variance_threshold']}")
print(f"  Threshold range:    [{CFG['threshold_min']}, {CFG['threshold_max']}] step={CFG['threshold_step']}")
print(f"  Pseudo pos pct:     {CFG['pseudo_pos_percentile']}%")
print(f"  Pseudo neg pct:     {CFG['pseudo_neg_percentile']}%")



  CONFIGURATION (CFG)
  Ensemble seeds:     [42, 123, 456]
  N folds:            5
  Stage 1 models:     45
  Total (both stages): 90
  Adv drop top:       5.0% features
  PCA variance keep:  0.95
  Threshold range:    [0.01, 0.5] step=0.005
  Pseudo pos pct:     99.0%
  Pseudo neg pct:     30.0%


# CELL 3: Data Loading & True Categorical Detection

**CRITICAL**: Only STRING-type features are categorical. The `≤2500 unique` heuristic from V4
is wrong — it catches 255 binary/count features that are truly numerical.


In [4]:
# ===================================================================
# CELL 3: Data Loading & True Categorical Detection
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 3: DATA LOADING & TRUE CATEGORICAL DETECTION")
print("=" * 70)

# Load data
train_raw = pd.read_csv(CFG['train_path'])
test_raw  = pd.read_csv(CFG['test_path'])
sub_raw   = pd.read_csv(CFG['sub_path'])

print(f"\n  Train shape: {train_raw.shape}")
print(f"  Test shape:  {test_raw.shape}")

# Target
y = train_raw['TARGET'].values.astype(int)
n_pos = np.sum(y == 1)
n_neg = np.sum(y == 0)
print(f"\n  Target distribution:")
print(f"    Class 0 (Stable):  {n_neg:,} ({100*n_neg/len(y):.2f}%)")
print(f"    Class 1 (At-Risk):  {n_pos:,} ({100*n_pos/len(y):.2f}%)")
print(f"    Imbalance ratio:    {n_neg/n_pos:.2f}:1")

# Feature columns
feature_cols = [c for c in train_raw.columns if c.startswith('feat_')]
print(f"\n  Total feature columns: {len(feature_cols)}")

# Detect string vs numerical features by dtype ONLY
string_cat_features = []
numerical_features = []

for c in feature_cols:
    if train_raw[c].dtype == 'object':
        string_cat_features.append(c)
    else:
        numerical_features.append(c)

print(f"\n  STRING-type categoricals: {len(string_cat_features)}")
for c in string_cat_features:
    print(f"    {c}: {train_raw[c].nunique():,} unique, samples={train_raw[c].dropna().iloc[:2].tolist()}")

print(f"\n  Numerical features: {len(numerical_features)}")

KNOWN_STRING_CATS = ['feat_142', 'feat_157', 'feat_318', 'feat_320', 'feat_325', 'feat_337']
assert set(string_cat_features) == set(KNOWN_STRING_CATS), \
    f"Cat mismatch! Expected {KNOWN_STRING_CATS}, got {string_cat_features}"
print(f"  ✓ All 6 known string cats confirmed.")

dbg.log_step("Data loaded", shape=train_raw.shape,
             extra=f"cats={len(string_cat_features)} (string-only), nums={len(numerical_features)}")



  CELL 3: DATA LOADING & TRUE CATEGORICAL DETECTION

  Train shape: (76020, 351)
  Test shape:  (60654, 351)

  Target distribution:
    Class 0 (Stable):  73,012 (96.04%)
    Class 1 (At-Risk):  3,008 (3.96%)
    Imbalance ratio:    24.27:1

  Total feature columns: 350

  STRING-type categoricals: 6
    feat_142: 2,333 unique, samples=['PRD_00490', 'PRD_01543']
    feat_157: 627 unique, samples=['PRV_294', 'PRV_035']
    feat_318: 12 unique, samples=['PERF_02', 'PERF_02']
    feat_320: 119 unique, samples=['CH_060', 'CH_069']
    feat_325: 1,710 unique, samples=['SEG_0327', 'SEG_0778']
    feat_337: 39 unique, samples=['OFC_39', 'OFC_08']

  Numerical features: 344
  ✓ All 6 known string cats confirmed.
  [D01] Data loaded  shape=(76020, 351)  +6.7s  cats=6 (string-only), nums=344


# CELL 4: Adversarial Validation — Detect & Drop Shifted Features

**STRATEGY**: Train a classifier to distinguish train from test. Features with high
importance in this model are covariate-shifted — they differ between train and test
distributions and will hurt generalization. We drop them BEFORE any feature engineering.


In [5]:
# ===================================================================
# CELL 4: Adversarial Validation
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 4: ADVERSARIAL VALIDATION (Train vs Test Drift Detection)")
print("=" * 70)

# Build adversarial dataset
# Use ONLY numerical features (categoricals handled separately)
train_num_raw = train_raw[numerical_features].fillna(0).astype(np.float32)
test_num_raw  = test_raw[numerical_features].fillna(0).astype(np.float32)

X_adv = np.vstack([train_num_raw.values, test_num_raw.values])
y_adv = np.hstack([np.zeros(len(train_raw)), np.ones(len(test_raw))])

print(f"  Adversarial dataset: {X_adv.shape}")
print(f"    Train samples (label=0): {len(train_raw):,}")
print(f"    Test samples  (label=1): {len(test_raw):,}")

# Train quick LightGBM to classify train vs test
print(f"\n  Training adversarial LGB ({CFG['adv_n_estimators']} trees)...")
t0 = time.time()

adv_model = LGBMClassifier(
    n_estimators=CFG['adv_n_estimators'],
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=CFG['seed'],
    verbosity=-1,
)
adv_model.fit(X_adv, y_adv)

# Evaluate adversarial detection
adv_preds = adv_model.predict_proba(X_adv)[:, 1]
adv_auc = np.mean((adv_preds[:len(train_raw)] < adv_preds[len(train_raw):].mean()))
# Simple metric: ROC AUC approximation
from sklearn.metrics import roc_auc_score
adv_roc = roc_auc_score(y_adv, adv_preds)
print(f"  Adversarial model ROC-AUC: {adv_roc:.4f}")
print(f"    AUC > 0.65 → significant drift detected")
print(f"    AUC ~ 0.50 → no drift (good!)")

# Get feature importance
adv_importance = adv_model.feature_importances_
feature_importance_pairs = list(zip(numerical_features, adv_importance))
feature_importance_pairs.sort(key=lambda x: x[1], reverse=True)

# Determine how many features to drop
n_drop = max(1, int(len(numerical_features) * CFG['adv_drop_top_pct'] / 100))
drop_threshold = feature_importance_pairs[n_drop][1]

drift_features = [f for f, imp in feature_importance_pairs[:n_drop]]
keep_numerical_features = [f for f in numerical_features if f not in drift_features]

print(f"\n  Adversarial Validation Results:")
print(f"    ROC-AUC: {adv_roc:.4f} ({'⚠️ DRIFT DETECTED' if adv_roc > 0.65 else '✅ No major drift'})")
print(f"    Dropping top {n_drop} features ({CFG['adv_drop_top_pct']}%)")
print(f"    Kept numerical features: {len(keep_numerical_features)} / {len(numerical_features)}")

print(f"\n  Top-10 drift features (DROPPED):")
for rank, (fname, imp) in enumerate(feature_importance_pairs[:10]):
    marker = "🔴 DROPPED" if fname in drift_features else "✅ KEPT"
    print(f"    {rank+1:2d}. {fname}: imp={imp:.4f}  {marker}")

if adv_roc > 0.70:
    print(f"\n  ⚠️  STRONG DRIFT DETECTED (AUC={adv_roc:.3f})")
    print(f"    Dropping {n_drop} most shifted features is critical for generalization.")

del adv_model, X_adv, y_adv, train_num_raw, test_num_raw; gc.collect()

dbg.log_step("Adversarial validation done",
             extra=f"Dropped {n_drop}/{len(numerical_features)} features, AUC={adv_roc:.4f}")



  CELL 4: ADVERSARIAL VALIDATION (Train vs Test Drift Detection)
  Adversarial dataset: (136674, 344)
    Train samples (label=0): 76,020
    Test samples  (label=1): 60,654

  Training adversarial LGB (200 trees)...
  Adversarial model ROC-AUC: 0.6865
    AUC > 0.65 → significant drift detected
    AUC ~ 0.50 → no drift (good!)

  Adversarial Validation Results:
    ROC-AUC: 0.6865 (⚠️ DRIFT DETECTED)
    Dropping top 17 features (5.0%)
    Kept numerical features: 327 / 344

  Top-10 drift features (DROPPED):
     1. feat_182: imp=624.0000  🔴 DROPPED
     2. feat_175: imp=486.0000  🔴 DROPPED
     3. feat_116: imp=295.0000  🔴 DROPPED
     4. feat_97: imp=282.0000  🔴 DROPPED
     5. feat_44: imp=248.0000  🔴 DROPPED
     6. feat_306: imp=226.0000  🔴 DROPPED
     7. feat_250: imp=167.0000  🔴 DROPPED
     8. feat_190: imp=153.0000  🔴 DROPPED
     9. feat_296: imp=149.0000  🔴 DROPPED
    10. feat_169: imp=146.0000  🔴 DROPPED
  [D02] Adversarial validation done  +8.3s  Dropped 17/344 featu

# CELL 5: Feature Engineering — QT + PCA(ddof=0) + Cat Encoding

**MATH FIX**: PCA explained_variance uses ddof=0 (division by n, not n-1).
This is implemented via manual SVD: `explained_variance = S² / n_samples`.

**Feature budget**: QT (kept numericals) → PCA(~95%) → ~100-150 components + 6 LE cats


In [6]:
# ===================================================================
# CELL 5: Feature Engineering
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 5: FEATURE ENGINEERING")
print("=" * 70)

# --- 5A: Extract features ---
X_train_num = train_raw[keep_numerical_features].fillna(0).astype(np.float32)
X_test_num  = test_raw[keep_numerical_features].fillna(0).astype(np.float32)

n_kept = X_train_num.shape[1]
print(f"\n  Kept numerical features: {n_kept} (dropped {len(drift_features)} shifted)")

# --- 5B: QuantileTransform ---
print(f"\n  --- QuantileTransformer(output='normal') ---")
qt = QuantileTransformer(
    output_distribution='normal',
    n_quantiles=min(1000, len(train_raw)),
    random_state=CFG['seed'],
    subsample=200_000
)

X_train_qt = qt.fit_transform(X_train_num).astype(np.float32)
X_test_qt  = qt.transform(X_test_num).astype(np.float32)

print(f"    QT train: {X_train_qt.shape}  range=[{X_train_qt.min():.3f}, {X_train_qt.max():.3f}]")
print(f"    QT test:  {X_test_qt.shape}  range=[{X_test_qt.min():.3f}, {X_test_qt.max():.3f}]")

dbg.log_step("QT done", shape=X_train_qt.shape)

# --- 5C: PCA with ddof=0 ---
# SKLEARN PCA uses ddof=1 (S²/(n-1)). We implement MANUAL SVD with ddof=0.
print(f"\n  --- PCA with ddof=0 (manual SVD, {CFG['pca_variance_threshold']:.0%} variance) ---")

X_qt_all = np.vstack([X_train_qt, X_test_qt])
n_total = X_qt_all.shape[0]

# Center the data
qt_mean = X_qt_all.mean(axis=0)
X_centered = X_qt_all - qt_mean

# SVD
print(f"    Computing SVD on {X_centered.shape}...")
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# ddof=0: divide by n (NOT n-1)
explained_variance_ddof0 = (S ** 2) / n_total
explained_variance_ratio = explained_variance_ddof0 / explained_variance_ddof0.sum()
cumsum_var = np.cumsum(explained_variance_ratio)

# Find n_components for target variance
n_pca = np.searchsorted(cumsum_var, CFG['pca_variance_threshold']) + 1
n_pca = min(n_pca, X_qt_all.shape[1])  # cap at original feature count

print(f"    Variance explained by top components (ddof=0):")
for k in [10, 25, 50, 100, n_pca]:
    if k <= len(cumsum_var):
        print(f"      Top-{k:3d}: {cumsum_var[k-1]:.4f} ({cumsum_var[k-1]*100:.1f}%)")

# Transform using selected components
X_train_pca = (U[:len(train_raw), :n_pca] * S[:n_pca]).astype(np.float32)
X_test_pca  = (U[len(train_raw):, :n_pca] * S[:n_pca]).astype(np.float32)

print(f"\n    PCA components kept: {n_pca}/{X_qt_all.shape[1]} "
      f"(retains {cumsum_var[n_pca-1]*100:.1f}% variance with ddof=0)")
print(f"    PCA train: {X_train_pca.shape}  range=[{X_train_pca.min():.3f}, {X_train_pca.max():.3f}]")
print(f"    PCA test:  {X_test_pca.shape}  range=[{X_test_pca.min():.3f}, {X_test_pca.max():.3f}]")

dbg.log_step("PCA done", shape=X_train_pca.shape, extra=f"{n_pca} components, ddof=0")

# Clean up large intermediates
del X_qt_all, X_centered, U, S, Vt; gc.collect()

# --- 5D: Row Statistics (on PCA components for efficiency) ---
if CFG['use_row_stats']:
    print(f"\n  --- Row Statistics (11 features, on PCA space) ---")
    row_stat_names = ['sum', 'mean', 'std', 'min', 'max', 'median',
                      'skew', 'kurtosis', 'n_zeros', 'mad', 'iqr']

    def compute_row_stats(arr):
        out = np.zeros((len(arr), 11), dtype=np.float32)
        out[:, 0] = np.sum(arr, axis=1)
        out[:, 1] = np.mean(arr, axis=1)
        out[:, 2] = np.std(arr, axis=1, ddof=0)   # ddof=0 FIX
        out[:, 3] = np.min(arr, axis=1)
        out[:, 4] = np.max(arr, axis=1)
        out[:, 5] = np.median(arr, axis=1)
        from scipy.stats import skew as sk_fn, kurtosis as kt_fn
        out[:, 6] = sk_fn(arr, axis=1)
        out[:, 7] = kt_fn(arr, axis=1)
        out[:, 8] = np.sum(arr == 0, axis=1).astype(np.float32)
        med = np.median(arr, axis=1, keepdims=True)
        out[:, 9] = np.median(np.abs(arr - med), axis=1)
        out[:, 10] = np.percentile(arr, 75, axis=1) - np.percentile(arr, 25, axis=1)
        return out

    row_stats_train = compute_row_stats(X_train_pca)
    row_stats_test  = compute_row_stats(X_test_pca)

    print(f"    Row stats train: {row_stats_train.shape}")
    print(f"    Row stats (ddof=0 for std): mean std={row_stats_train[:, 2].mean():.4f}")

    dbg.log_step("Row stats done", shape=row_stats_train.shape)
else:
    row_stats_train = np.zeros((len(train_raw), 0), dtype=np.float32)
    row_stats_test  = np.zeros((len(test_raw), 0), dtype=np.float32)
    row_stat_names = []

n_rs = row_stats_train.shape[1]

# --- 5E: KMeans Clustering ---
if CFG['use_kmeans']:
    print(f"\n  --- KMeans Clustering (k={CFG['kmeans_clusters']}) ---")
    from sklearn.cluster import KMeans
    kmeans_train = np.zeros((len(train_raw), len(CFG['kmeans_clusters'])), dtype=np.int32)
    kmeans_test  = np.zeros((len(test_raw), len(CFG['kmeans_clusters'])), dtype=np.int32)

    for i, k in enumerate(CFG['kmeans_clusters']):
        km = KMeans(n_clusters=k, random_state=CFG['seed'] + i, n_init=10, max_iter=300)
        km.fit(X_train_pca)
        kmeans_train[:, i] = km.predict(X_train_pca)
        kmeans_test[:, i]  = km.predict(X_test_pca)
        print(f"    k={k}: inertia={km.inertia_:.1f}")

    kmeans_col_names = [f'kmeans_k{k}' for k in CFG['kmeans_clusters']]
    print(f"    KMeans features: {kmeans_train.shape[1]}")

    dbg.log_step("KMeans done", shape=kmeans_train.shape)
else:
    kmeans_train = np.zeros((len(train_raw), 0), dtype=np.int32)
    kmeans_test  = np.zeros((len(test_raw), 0), dtype=np.int32)
    kmeans_col_names = []

n_km = kmeans_train.shape[1]

# --- 5F: Categorical Encoding (6 TRUE String Cats Only) ---
print(f"\n  --- Categorical Encoding ({len(string_cat_features)} cats) ---")

# Raw strings for CatBoost
train_cats_raw = pd.DataFrame(index=train_raw.index)
test_cats_raw  = pd.DataFrame(index=test_raw.index)
for c in string_cat_features:
    train_cats_raw[c] = train_raw[c].fillna('MISSING').astype(str)
    test_cats_raw[c]  = test_raw[c].fillna('MISSING').astype(str)

# Label Encoding for XGB/LGB
train_cats_le = pd.DataFrame(index=train_raw.index)
test_cats_le  = pd.DataFrame(index=test_raw.index)
label_encoders = {}

for c in string_cat_features:
    le = LabelEncoder()
    all_vals = np.concatenate([train_cats_raw[c].values, test_cats_raw[c].values])
    le.fit(all_vals)
    label_encoders[c] = le
    train_cats_le[f'{c}_le'] = le.transform(train_cats_raw[c].values).astype(np.int32)
    test_cats_le[f'{c}_le']  = le.transform(test_cats_raw[c].values).astype(np.int32)

print(f"    Label-encoded cats: {train_cats_le.shape[1]} features")
for col in train_cats_le.columns:
    print(f"      {col}: unique={train_cats_le[col].nunique()}, "
          f"range=[{train_cats_le[col].min()}, {train_cats_le[col].max()}]")

n_cat = train_cats_le.shape[1]

dbg.log_step("Cat encoding done", extra=f"{n_cat} LE features for {len(string_cat_features)} cats")



  CELL 5: FEATURE ENGINEERING

  Kept numerical features: 327 (dropped 17 shifted)

  --- QuantileTransformer(output='normal') ---
    QT train: (76020, 327)  range=[-5.199, 5.199]
    QT test:  (60654, 327)  range=[-5.199, 5.199]
  [D03] QT done  shape=(76020, 327)  +5.0s

  --- PCA with ddof=0 (manual SVD, 95% variance) ---
    Computing SVD on (136674, 327)...
    Variance explained by top components (ddof=0):
      Top- 10: 0.7750 (77.5%)
      Top- 25: 0.8968 (89.7%)
      Top- 50: 0.9562 (95.6%)
      Top-100: 0.9978 (99.8%)
      Top- 47: 0.9506 (95.1%)

    PCA components kept: 47/327 (retains 95.1% variance with ddof=0)
    PCA train: (76020, 47)  range=[-48.201, 53.565]
    PCA test:  (60654, 47)  range=[-49.418, 54.023]
  [D04] PCA done  shape=(76020, 47)  +4.8s  47 components, ddof=0

  --- Row Statistics (11 features, on PCA space) ---
    Row stats train: (76020, 11)
    Row stats (ddof=0 for std): mean std=2.8338
  [D05] Row stats done  shape=(76020, 11)  +1.0s

  --- K

# CELL 6: Feature Assembly — Dual Pipelines

**Pipeline A (XGB/LGB)**: numpy float32 array — PCA components + row stats + LE cats + KMeans
**Pipeline B (CatBoost)**: DataFrame with raw string cats + KMeans as categorical features


In [7]:
# ===================================================================
# CELL 6: Feature Assembly
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 6: FEATURE ASSEMBLY")
print("=" * 70)

# --- Pipeline A: XGBoost/LightGBM (all float32 numpy) ---
X_tr_parts = [X_train_pca]
X_te_parts = [X_test_pca]
col_map = {'pca': (0, n_pca)}
current_col = n_pca

# Row stats
if row_stats_train.shape[1] > 0:
    X_tr_parts.append(row_stats_train)
    X_te_parts.append(row_stats_test)
    col_map['row_stat'] = (current_col, current_col + row_stats_train.shape[1])
    current_col += row_stats_train.shape[1]

# KMeans (as float32)
if kmeans_train.shape[1] > 0:
    X_tr_parts.append(kmeans_train.astype(np.float32))
    X_te_parts.append(kmeans_test.astype(np.float32))
    col_map['kmeans'] = (current_col, current_col + kmeans_train.shape[1])
    current_col += kmeans_train.shape[1]

# Label-encoded cats (as float32)
if train_cats_le.shape[1] > 0:
    X_tr_parts.append(train_cats_le.values.astype(np.float32))
    X_te_parts.append(test_cats_le.values.astype(np.float32))
    col_map['cat_le'] = (current_col, current_col + train_cats_le.shape[1])
    current_col += train_cats_le.shape[1]

X_tr = np.hstack(X_tr_parts).astype(np.float32)
X_te = np.hstack(X_te_parts).astype(np.float32)

n_total = X_tr.shape[1]
print(f"\n  Pipeline A (XGB/LGB): numpy float32")
print(f"    Train: {X_tr.shape}  dtype={X_tr.dtype}")
print(f"    Test:  {X_te.shape}  dtype={X_te.dtype}")
comp_parts = [f"{v[1]-v[0]} {k}" for k, v in col_map.items()]
print(f"    Composition: {' + '.join(comp_parts)} = {n_total}")

# --- Pipeline B: CatBoost DataFrame ---
cb_train = pd.DataFrame(X_train_pca, columns=[f'pca_{i}' for i in range(n_pca)])
cb_test  = pd.DataFrame(X_test_pca,  columns=[f'pca_{i}' for i in range(n_pca)])

cb_cat_indices = []

# Add raw string cat features (CatBoost handles natively)
for c in string_cat_features:
    col_idx = cb_train.shape[1]
    cb_train[c] = train_cats_raw[c].values
    cb_test[c]  = test_cats_raw[c].values
    cb_cat_indices.append(col_idx)

# Row stats (numerical)
for i in range(row_stats_train.shape[1]):
    cb_train[f'row_stat_{i}'] = row_stats_train[:, i]
    cb_test[f'row_stat_{i}']  = row_stats_test[:, i]

# KMeans (as string for CatBoost categorical)
if kmeans_train.shape[1] > 0:
    for i, k in enumerate(CFG['kmeans_clusters']):
        col_idx = cb_train.shape[1]
        col_name = f'kmeans_k{k}'
        cb_train[col_name] = kmeans_train[:, i].astype(str)
        cb_test[col_name]  = kmeans_test[:, i].astype(str)
        cb_cat_indices.append(col_idx)

print(f"\n  Pipeline B (CatBoost): DataFrame")
print(f"    Train: {cb_train.shape}  Test: {cb_test.shape}")
print(f"    cat_features count: {len(cb_cat_indices)} ({len(string_cat_features)} cats + {n_km} KMeans)")
print(f"    cat_features names: {list(cb_train.columns[cb_cat_indices])}")

# Verify string dtype for cat columns
for ci in cb_cat_indices:
    col_name = cb_train.columns[ci]
    assert cb_train[col_name].dtype == 'object', \
        f"CatBoost cat column '{col_name}' must be object, got {cb_train[col_name].dtype}"
print(f"    ✓ All {len(cb_cat_indices)} cat columns verified as object dtype")

# Free memory
del X_train_pca, X_test_pca, X_train_qt, X_test_qt, X_train_num, X_test_num
del X_tr_parts, X_te_parts; gc.collect()

dbg.log_step("Feature assembly done", shape=X_tr.shape,
             extra=f"{n_total} features, {len(cb_cat_indices)} CB cats")



  CELL 6: FEATURE ASSEMBLY

  Pipeline A (XGB/LGB): numpy float32
    Train: (76020, 66)  dtype=float32
    Test:  (60654, 66)  dtype=float32
    Composition: 47 pca + 11 row_stat + 2 kmeans + 6 cat_le = 66

  Pipeline B (CatBoost): DataFrame
    Train: (76020, 66)  Test: (60654, 66)
    cat_features count: 8 (6 cats + 2 KMeans)
    cat_features names: ['feat_142', 'feat_157', 'feat_318', 'feat_320', 'feat_325', 'feat_337', 'kmeans_k8', 'kmeans_k16']
    ✓ All 8 cat columns verified as object dtype
  [D08] Feature assembly done  shape=(76020, 66)  +0.3s  66 features, 8 CB cats


# CELL 7: Stage 1 — Pure LogLoss Model Training (3 Models × 3 Seeds × 5 Folds)

**NO SMOTE. NO class weights. NO auto_class_weights.**
Models learn raw calibrated probabilities from the natural 24:1 imbalance.
We find the right threshold LATER via grid search on OOF.


In [8]:
# ===================================================================
# CELL 7: STAGE 1 — Pure LogLoss Model Training
# ===================================================================
print("\n" + "=" * 70)
print(f"  CELL 7: STAGE 1 — PURE LOGLOSS TRAINING "
      f"({len(CFG['ensemble_seeds'])} seeds × {CFG['n_folds']} folds × 3 models = "
      f"{len(CFG['ensemble_seeds']) * CFG['n_folds'] * 3} models)")
print("=" * 70)
print("  🔑 KEY: NO SMOTE, NO class weights — pure LogLoss only")

n_train = len(train_raw)
n_test  = len(test_raw)
n_seeds = len(CFG['ensemble_seeds'])

# OOF and test prediction storage
oof_xgb_s1 = np.zeros((n_train, n_seeds), dtype=np.float32)
oof_cb_s1  = np.zeros((n_train, n_seeds), dtype=np.float32)
oof_lgb_s1 = np.zeros((n_train, n_seeds), dtype=np.float32)

test_xgb_s1 = np.zeros((n_test, n_seeds), dtype=np.float32)
test_cb_s1  = np.zeros((n_test, n_seeds), dtype=np.float32)
test_lgb_s1 = np.zeros((n_test, n_seeds), dtype=np.float32)

fold_scores_s1 = {'xgb': {s: [] for s in CFG['ensemble_seeds']},
                   'cb':  {s: [] for s in CFG['ensemble_seeds']},
                   'lgb': {s: [] for s in CFG['ensemble_seeds']}}

for seed_idx, seed in enumerate(CFG['ensemble_seeds']):
    print(f"\n{'='*60}")
    print(f"  ENSEMBLE SEED {seed} ({seed_idx+1}/{n_seeds})")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y)):
        t_fold_start = time.time()
        print(f"\n  --- Fold {fold+1}/{CFG['n_folds']} ---")

        # NO SMOTE — use raw fold data
        X_tr_fold = X_tr[tr_idx]
        X_va_fold = X_tr[va_idx]
        y_tr_fold = y[tr_idx]
        y_va_fold = y[va_idx]

        # CatBoost fold data
        cb_tr_fold = cb_train.iloc[tr_idx].reset_index(drop=True)
        cb_va_fold = cb_train.iloc[va_idx].reset_index(drop=True)

        print(f"    Train: {X_tr_fold.shape[0]:,} rows, pos={np.sum(y_tr_fold==1)} "
              f"({100*np.mean(y_tr_fold==1):.2f}%)")
        print(f"    Valid: {X_va_fold.shape[0]:,} rows, pos={np.sum(y_va_fold==1)} "
              f"({100*np.mean(y_va_fold==1):.2f}%)")

        # ================================================================
        # XGBOOST — Pure LogLoss, NO scale_pos_weight
        # ================================================================
        t0 = time.time()
        xgb_params = CFG['xgb_params'].copy()
        xgb_params['random_state'] = seed

        xgb = XGBClassifier(**xgb_params)
        xgb.fit(X_tr_fold, y_tr_fold,
                eval_set=[(X_va_fold, y_va_fold)],
                verbose=False)

        oof_xgb_s1[va_idx, seed_idx] = xgb.predict_proba(X_va_fold)[:, 1]
        test_xgb_s1[:, seed_idx] += xgb.predict_proba(X_te)[:, 1] / CFG['n_folds']

        # Evaluate at MULTIPLE thresholds (raw probs, F1 depends on cutoff)
        va_probs = oof_xgb_s1[va_idx, seed_idx]
        # Quick check at 0.5 just for logging
        f1_05 = f1_score(y_va_fold, (va_probs >= 0.5).astype(int))
        pos_05 = np.mean(va_probs >= 0.5) * 100
        mean_prob = np.mean(va_probs)
        fold_scores_s1['xgb'][seed].append(f1_05)
        dt = time.time() - t0
        print(f"    [XGBoost seed={seed}] F1@0.5={f1_05:.5f}  mean_prob={mean_prob:.4f}  "
              f"pos@0.5={pos_05:.1f}%  [{dt:.0f}s]")

        del xgb; gc.collect()

        # ================================================================
        # CATBOOST — Pure LogLoss, NO auto_class_weights
        # ================================================================
        t0 = time.time()
        cb_params = CFG['catboost_params'].copy()
        cb_params['random_seed'] = seed
        # DELIBERATELY OMITTED: auto_class_weights, scale_pos_weight

        cb = CatBoostClassifier(**cb_params, cat_features=cb_cat_indices)
        cb.fit(cb_tr_fold, y_tr_fold,
               eval_set=(cb_va_fold, y_va_fold),
               verbose=False)

        oof_cb_s1[va_idx, seed_idx] = cb.predict_proba(cb_va_fold)[:, 1]
        test_cb_s1[:, seed_idx] += cb.predict_proba(cb_test)[:, 1] / CFG['n_folds']

        va_probs_cb = oof_cb_s1[va_idx, seed_idx]
        f1_05_cb = f1_score(y_va_fold, (va_probs_cb >= 0.5).astype(int))
        pos_05_cb = np.mean(va_probs_cb >= 0.5) * 100
        mean_prob_cb = np.mean(va_probs_cb)
        fold_scores_s1['cb'][seed].append(f1_05_cb)
        dt_cb = time.time() - t0
        print(f"    [CatBoost seed={seed}] F1@0.5={f1_05_cb:.5f}  "
              f"mean_prob={mean_prob_cb:.4f}  pos@0.5={pos_05_cb:.1f}%  [{dt_cb:.0f}s]")

        del cb; gc.collect()

        # ================================================================
        # LIGHTGBM — Pure LogLoss, NO scale_pos_weight
        # ================================================================
        t0 = time.time()
        lgb_params = CFG['lgb_params'].copy()
        lgb_params['random_state'] = seed

        lgb = LGBMClassifier(**lgb_params)
        lgb.fit(X_tr_fold, y_tr_fold,
                eval_set=[(X_va_fold, y_va_fold)],
                eval_metric='logloss',
                callbacks=[early_stopping(200), log_evaluation(0)])

        oof_lgb_s1[va_idx, seed_idx] = lgb.predict_proba(X_va_fold)[:, 1]
        test_lgb_s1[:, seed_idx] += lgb.predict_proba(X_te)[:, 1] / CFG['n_folds']

        va_probs_lgb = oof_lgb_s1[va_idx, seed_idx]
        f1_05_lgb = f1_score(y_va_fold, (va_probs_lgb >= 0.5).astype(int))
        pos_05_lgb = np.mean(va_probs_lgb >= 0.5) * 100
        mean_prob_lgb = np.mean(va_probs_lgb)
        fold_scores_s1['lgb'][seed].append(f1_05_lgb)
        dt_lgb = time.time() - t0
        print(f"    [LightGBM seed={seed}] F1@0.5={f1_05_lgb:.5f}  "
              f"mean_prob={mean_prob_lgb:.4f}  pos@0.5={pos_05_lgb:.1f}%  [{dt_lgb:.0f}s]")

        del lgb; gc.collect()

        t_fold_total = time.time() - t_fold_start
        print(f"    Fold total: {t_fold_total:.0f}s")

    # End of folds for this seed — quick status
    s1_f1_xgb = f1_score(y, (np.nan_to_num(oof_xgb_s1[:, seed_idx], 0) >= 0.5).astype(int))
    s1_f1_cb  = f1_score(y, (np.nan_to_num(oof_cb_s1[:, seed_idx], 0) >= 0.5).astype(int))
    s1_f1_lgb = f1_score(y, (np.nan_to_num(oof_lgb_s1[:, seed_idx], 0) >= 0.5).astype(int))
    print(f"\n  Seed {seed} cumulative OOF@0.5: XGB={s1_f1_xgb:.5f}  CB={s1_f1_cb:.5f}  LGB={s1_f1_lgb:.5f}")

elapsed_s1 = time.time() - T_START
print(f"\n{'='*60}")
print(f"  STAGE 1 COMPLETE — {len(CFG['ensemble_seeds'])*CFG['n_folds']*3} models trained")
print(f"  Stage 1 elapsed: {elapsed_s1/60:.1f} min ({elapsed_s1:.0f}s)")
print(f"{'='*60}")



  CELL 7: STAGE 1 — PURE LOGLOSS TRAINING (3 seeds × 5 folds × 3 models = 45 models)
  🔑 KEY: NO SMOTE, NO class weights — pure LogLoss only

  ENSEMBLE SEED 42 (1/3)

  --- Fold 1/5 ---
    Train: 60,816 rows, pos=2407 (3.96%)
    Valid: 15,204 rows, pos=601 (3.95%)
    [XGBoost seed=42] F1@0.5=0.04452  mean_prob=0.0382  pos@0.5=0.2%  [13s]
    [CatBoost seed=42] F1@0.5=0.07244  mean_prob=0.0377  pos@0.5=0.2%  [110s]
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[504]	valid_0's binary_logloss: 0.138166
    [LightGBM seed=42] F1@0.5=0.04167  mean_prob=0.0382  pos@0.5=0.2%  [11s]
    Fold total: 134s

  --- Fold 2/5 ---
    Train: 60,816 rows, pos=2407 (3.96%)
    Valid: 15,204 rows, pos=601 (3.95%)
    [XGBoost seed=42] F1@0.5=0.08138  mean_prob=0.0391  pos@0.5=0.2%  [11s]
    [CatBoost seed=42] F1@0.5=0.09020  mean_prob=0.0381  pos@0.5=0.3%  [120s]
Training until validation scores don't improve for 200 rounds
Early stopping, best ite

# CELL 8: Threshold Optimization — Find Optimal F1 Cutoff on OOF

**WHY**: With pure LogLoss on 4% positive data, models predict ~0.04 mean probability.
A 0.5 threshold would give all-zero predictions. We grid-search from 0.01 to 0.50
to find the EXACT threshold that maximizes F1 Score on OOF predictions.

This threshold WILL be used to convert test probabilities to binary 0/1 for submission.


In [9]:
# ===================================================================
# CELL 8: Threshold Optimization on OOF
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 8: THRESHOLD OPTIMIZATION ON OOF")
print("=" * 70)

# Average OOF across seeds for each model
xgb_oof_avg = np.nan_to_num(np.mean(oof_xgb_s1, axis=1), 0)
cb_oof_avg  = np.nan_to_num(np.mean(oof_cb_s1, axis=1), 0)
lgb_oof_avg = np.nan_to_num(np.mean(oof_lgb_s1, axis=1), 0)

# Individual seed OOF for ensemble averaging
all_oof_seeds = np.hstack([oof_xgb_s1, oof_cb_s1, oof_lgb_s1])
oof_uniform = np.nan_to_num(np.mean(all_oof_seeds, axis=1), 0)

print(f"\n  OOF prediction stats (raw probabilities):")
for label, arr in [('XGBoost', xgb_oof_avg), ('CatBoost', cb_oof_avg),
                    ('LightGBM', lgb_oof_avg), ('Uniform Ensemble', oof_uniform)]:
    print(f"    {label:20s}: mean={np.mean(arr):.5f}  med={np.median(arr):.5f}  "
          f"max={np.max(arr):.4f}  p95={np.percentile(arr, 95):.4f}  p99={np.percentile(arr, 99):.4f}")

# --- Threshold Grid Search ---
thresholds = np.arange(CFG['threshold_min'], CFG['threshold_max'] + CFG['threshold_step']/2,
                        CFG['threshold_step'])

print(f"\n  Grid searching {len(thresholds)} thresholds [{CFG['threshold_min']:.3f}, "
      f"{CFG['threshold_max']:.3f}] step={CFG['threshold_step']:.3f}...")

best_results = {}

for label, oof_arr in [
    ('XGBoost', xgb_oof_avg),
    ('CatBoost', cb_oof_avg),
    ('LightGBM', lgb_oof_avg),
    ('Uniform Ensemble', oof_uniform),
]:
    best_f1 = 0
    best_t = 0.5
    best_prec = 0
    best_rec = 0
    best_pos = 0

    for t in thresholds:
        binary = (oof_arr >= t).astype(int)
        if np.sum(binary) == 0:
            continue  # skip all-zero
        f1 = f1_score(y, binary)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            best_prec = precision_score(y, binary, zero_division=0)
            best_rec = recall_score(y, binary, zero_division=0)
            best_pos = np.mean(binary) * 100

    best_results[label] = {
        'threshold': best_t, 'f1': best_f1,
        'precision': best_prec, 'recall': best_rec,
        'pos_rate': best_pos,
    }

# --- Display Results ---
print(f"\n  {'='*70}")
print(f"  OPTIMAL THRESHOLDS (maximizing OOF F1)")
print(f"  {'='*70}")
print(f"  {'Model':20s} {'Best Thresh':>12s} {'F1':>10s} {'Precision':>10s} {'Recall':>10s} {'Pos%':>8s}")
print(f"  {'-'*70}")

for label, res in best_results.items():
    print(f"  {label:20s} {res['threshold']:12.4f} {res['f1']:10.5f} "
          f"{res['precision']:10.4f} {res['recall']:10.4f} {res['pos_rate']:7.2f}%")

# --- Select Best Ensemble ---
# Compare all candidates: pick the one with best OOF F1
best_ensemble_label = 'Uniform Ensemble'
best_ensemble_oof = oof_uniform
best_ensemble_f1 = best_results['Uniform Ensemble']['f1']
best_ensemble_threshold = best_results['Uniform Ensemble']['threshold']

for label, res in best_results.items():
    if res['f1'] > best_ensemble_f1:
        best_ensemble_f1 = res['f1']
        best_ensemble_label = label
        best_ensemble_threshold = res['threshold']
        # map label back to oof array
        if 'XGBoost' in label and 'CatBoost' not in label and 'LightGBM' not in label:
            best_ensemble_oof = xgb_oof_avg
        elif 'CatBoost' in label:
            best_ensemble_oof = cb_oof_avg
        elif 'LightGBM' in label:
            best_ensemble_oof = lgb_oof_avg
        elif 'Uniform' in label:
            best_ensemble_oof = oof_uniform

print(f"\n  ▶ SELECTED: {best_ensemble_label}")
print(f"    Optimal threshold: {best_ensemble_threshold:.4f}")
print(f"    OOF F1:            {best_ensemble_f1:.5f}")
print(f"    (Compare: F1@0.5 = {f1_score(y, (best_ensemble_oof >= 0.5).astype(int)):.5f} — "
      f"threshold optimization gains {best_ensemble_f1 - f1_score(y, (best_ensemble_oof >= 0.5).astype(int)):.4f})")

# Store globally for pseudo-labeling and submission
OPTIMAL_THRESHOLD = best_ensemble_threshold
BEST_OOF_F1_S1 = best_ensemble_f1

dbg.log_step("Threshold optimization done",
             extra=f"optimal={OPTIMAL_THRESHOLD:.4f}, OOF F1={BEST_OOF_F1_S1:.5f}")



  CELL 8: THRESHOLD OPTIMIZATION ON OOF

  OOF prediction stats (raw probabilities):
    XGBoost             : mean=0.03828  med=0.01672  max=0.8041  p95=0.1474  p99=0.3181
    CatBoost            : mean=0.03726  med=0.01671  max=0.9095  p95=0.1417  p99=0.3108
    LightGBM            : mean=0.03801  med=0.01706  max=0.8268  p95=0.1446  p99=0.3074
    Uniform Ensemble    : mean=0.03785  med=0.01698  max=0.8389  p95=0.1441  p99=0.3094

  Grid searching 99 thresholds [0.010, 0.500] step=0.005...

  OPTIMAL THRESHOLDS (maximizing OOF F1)
  Model                 Best Thresh         F1  Precision     Recall     Pos%
  ----------------------------------------------------------------------
  XGBoost                    0.1300    0.28811     0.2375     0.3660    6.10%
  CatBoost                   0.1400    0.30150     0.2679     0.3447    5.09%
  LightGBM                   0.1400    0.29010     0.2535     0.3391    5.29%
  Uniform Ensemble           0.1400    0.29365     0.2578     0.3411    5.

# CELL 9: Pseudo-Labeling — The Winning Trick

**STRATEGY**:
1. Get test predictions from Stage 1 ensemble
2. Select HIGH-CONFIDENCE test samples using adaptive percentile thresholds
   - Pseudo-positive: top 1% highest predicted probabilities
   - Pseudo-negative: bottom 30% lowest predicted probabilities
3. Append pseudo-labeled samples to training data
4. Retrain ALL models from scratch on augmented data
5. This gives models more data AND exposes them to test-distribution patterns


In [10]:
# ===================================================================
# CELL 9: Pseudo-Labeling
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 9: PSEUDO-LABELING")
print("=" * 70)

# --- 9A: Get Stage 1 test predictions ---
test_xgb_avg_s1 = np.mean(test_xgb_s1, axis=1)
test_cb_avg_s1  = np.mean(test_cb_s1, axis=1)
test_lgb_avg_s1 = np.mean(test_lgb_s1, axis=1)
test_ensemble_s1 = np.mean(np.hstack([test_xgb_s1, test_cb_s1, test_lgb_s1]), axis=1)

print(f"\n  Stage 1 test prediction stats:")
print(f"    Ensemble mean: {np.mean(test_ensemble_s1):.5f}  "
      f"med={np.median(test_ensemble_s1):.5f}  "
      f"max={np.max(test_ensemble_s1):.4f}")
print(f"    p90={np.percentile(test_ensemble_s1, 90):.4f}  "
      f"p95={np.percentile(test_ensemble_s1, 95):.4f}  "
      f"p99={np.percentile(test_ensemble_s1, 99):.4f}")

# --- 9B: Select pseudo-labels using adaptive thresholds ---
pos_threshold = np.percentile(test_ensemble_s1, CFG['pseudo_pos_percentile'])
neg_threshold = np.percentile(test_ensemble_s1, CFG['pseudo_neg_percentile'])

print(f"\n  Pseudo-label thresholds:")
print(f"    Class 1: prob >= {pos_threshold:.4f} (top {100-CFG['pseudo_pos_percentile']:.1f}%)")
print(f"    Class 0: prob <= {neg_threshold:.4f} (bottom {CFG['pseudo_neg_percentile']:.1f}%)")

pseudo_pos_mask = test_ensemble_s1 >= pos_threshold
pseudo_neg_mask = test_ensemble_s1 <= neg_threshold

n_pseudo_pos = np.sum(pseudo_pos_mask)
n_pseudo_neg = np.sum(pseudo_neg_mask)

print(f"\n  Pseudo-labels selected:")
print(f"    Class 1 (pos): {n_pseudo_pos:,} samples "
      f"(mean prob={np.mean(test_ensemble_s1[pseudo_pos_mask]):.4f})")
print(f"    Class 0 (neg): {n_pseudo_neg:,} samples "
      f"(mean prob={np.mean(test_ensemble_s1[pseudo_neg_mask]):.4f})")

if n_pseudo_pos < CFG['pseudo_min_samples']:
    print(f"    ⚠️  Too few pseudo-positives ({n_pseudo_pos} < {CFG['pseudo_min_samples']}). "
          f"Using top {CFG['pseudo_min_samples']} instead.")
    top_indices = np.argsort(test_ensemble_s1)[-CFG['pseudo_min_samples']:]
    pseudo_pos_mask = np.zeros(n_test, dtype=bool)
    pseudo_pos_mask[top_indices] = True
    n_pseudo_pos = CFG['pseudo_min_samples']
    pos_threshold = test_ensemble_s1[top_indices[0]]
    print(f"    Adjusted pos threshold: {pos_threshold:.4f}")

# --- 9C: Create augmented training data ---
# Original train data
X_tr_aug = X_tr.copy()
y_aug = y.copy()

# For CatBoost
cb_train_aug = cb_train.copy()

# Append pseudo-labeled test samples
# Pseudo-positive samples
if n_pseudo_pos > 0:
    X_tr_aug = np.vstack([X_tr_aug, X_te[pseudo_pos_mask]])
    y_aug = np.hstack([y_aug, np.ones(n_pseudo_pos, dtype=int)])
    cb_train_aug = pd.concat([cb_train_aug, cb_test.iloc[pseudo_pos_mask].reset_index(drop=True)],
                              ignore_index=True)

# Pseudo-negative samples (use a subset to avoid overwhelming)
# Cap pseudo-negatives at 2× original train size to maintain reasonable ratio
max_pseudo_neg = min(n_pseudo_neg, len(train_raw))
if n_pseudo_neg > max_pseudo_neg:
    neg_indices = np.where(pseudo_neg_mask)[0]
    # Randomly sample
    np.random.seed(CFG['seed'])
    selected_neg = np.random.choice(neg_indices, size=max_pseudo_neg, replace=False)
    pseudo_neg_mask_final = np.zeros(n_test, dtype=bool)
    pseudo_neg_mask_final[selected_neg] = True
    n_pseudo_neg = max_pseudo_neg
else:
    pseudo_neg_mask_final = pseudo_neg_mask

if n_pseudo_neg > 0:
    X_tr_aug = np.vstack([X_tr_aug, X_te[pseudo_neg_mask_final]])
    y_aug = np.hstack([y_aug, np.zeros(n_pseudo_neg, dtype=int)])
    cb_train_aug = pd.concat([cb_train_aug, cb_test.iloc[pseudo_neg_mask_final].reset_index(drop=True)],
                              ignore_index=True)

n_aug = len(y_aug)
n_orig = len(y)
aug_pos_rate = np.mean(y_aug == 1) * 100

print(f"\n  Augmented training data:")
print(f"    Original: {n_orig:,} rows (pos: {np.sum(y==1):,}, {100*np.mean(y==1):.2f}%)")
print(f"    Augmented: {n_aug:,} rows (pos: {np.sum(y_aug==1):,}, {aug_pos_rate:.2f}%)")
print(f"    Added: {n_aug - n_orig:,} pseudo-labeled samples "
      f"({n_pseudo_pos} pos + {n_pseudo_neg} neg)")
print(f"    Test samples unlabeled: {n_test - n_pseudo_pos - n_pseudo_neg:,} / {n_test:,}")

# Track which samples are original vs pseudo-labeled (for clean OOF later)
is_original = np.zeros(n_aug, dtype=bool)
is_original[:n_orig] = True

dbg.log_step("Pseudo-labeling done",
             extra=f"+{n_pseudo_pos} pos + {n_pseudo_neg} neg → {n_aug:,} total")



  CELL 9: PSEUDO-LABELING

  Stage 1 test prediction stats:
    Ensemble mean: 0.03594  med=0.01704  max=0.8367
    p90=0.0867  p95=0.1330  p99=0.2706

  Pseudo-label thresholds:
    Class 1: prob >= 0.2706 (top 1.0%)
    Class 0: prob <= 0.0095 (bottom 30.0%)

  Pseudo-labels selected:
    Class 1 (pos): 607 samples (mean prob=0.3641)
    Class 0 (neg): 18,196 samples (mean prob=0.0058)

  Augmented training data:
    Original: 76,020 rows (pos: 3,008, 3.96%)
    Augmented: 94,823 rows (pos: 3,615, 3.81%)
    Added: 18,803 pseudo-labeled samples (607 pos + 18196 neg)
    Test samples unlabeled: 41,851 / 60,654
  [D10] Pseudo-labeling done  +0.2s  +607 pos + 18196 neg → 94,823 total


# CELL 10: Stage 2 — Retrain on Augmented Data

Retrain ALL models from scratch on the augmented dataset (original + pseudo-labels).
Use the SAME 5-fold CV strategy but now on the larger dataset.
Track OOF only on ORIGINAL samples for unbiased threshold optimization.


In [11]:
# ===================================================================
# CELL 10: STAGE 2 — Retrain on Augmented Data
# ===================================================================
print("\n" + "=" * 70)
print(f"  CELL 10: STAGE 2 — RETRAIN ON AUGMENTED DATA")
print("=" * 70)

oof_xgb_s2 = np.zeros((n_aug, n_seeds), dtype=np.float32)
oof_cb_s2  = np.zeros((n_aug, n_seeds), dtype=np.float32)
oof_lgb_s2 = np.zeros((n_aug, n_seeds), dtype=np.float32)

test_xgb_s2 = np.zeros((n_test, n_seeds), dtype=np.float32)
test_cb_s2  = np.zeros((n_test, n_seeds), dtype=np.float32)
test_lgb_s2 = np.zeros((n_test, n_seeds), dtype=np.float32)

fold_scores_s2 = {'xgb': {s: [] for s in CFG['ensemble_seeds']},
                   'cb':  {s: [] for s in CFG['ensemble_seeds']},
                   'lgb': {s: [] for s in CFG['ensemble_seeds']}}

for seed_idx, seed in enumerate(CFG['ensemble_seeds']):
    print(f"\n{'='*60}")
    print(f"  ENSEMBLE SEED {seed} ({seed_idx+1}/{n_seeds}) — AUGMENTED DATA")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_aug, y_aug)):
        t_fold_start = time.time()
        print(f"\n  --- Fold {fold+1}/{CFG['n_folds']} ---")

        X_tr_fold = X_tr_aug[tr_idx]
        X_va_fold = X_tr_aug[va_idx]
        y_tr_fold = y_aug[tr_idx]
        y_va_fold = y_aug[va_idx]

        cb_tr_fold = cb_train_aug.iloc[tr_idx].reset_index(drop=True)
        cb_va_fold = cb_train_aug.iloc[va_idx].reset_index(drop=True)

        n_pseudo_in_fold = np.sum(~is_original[tr_idx])
        n_orig_in_fold = np.sum(is_original[tr_idx])
        print(f"    Train: {X_tr_fold.shape[0]:,} rows ({n_orig_in_fold:,} orig + {n_pseudo_in_fold:,} pseudo), "
              f"pos={np.sum(y_tr_fold==1):,} ({100*np.mean(y_tr_fold==1):.2f}%)")

        # ================================================================
        # XGBOOST
        # ================================================================
        t0 = time.time()
        xgb_params = CFG['xgb_params'].copy()
        xgb_params['random_state'] = seed

        xgb = XGBClassifier(**xgb_params)
        xgb.fit(X_tr_fold, y_tr_fold,
                eval_set=[(X_va_fold, y_va_fold)],
                verbose=False)

        oof_xgb_s2[va_idx, seed_idx] = xgb.predict_proba(X_va_fold)[:, 1]
        test_xgb_s2[:, seed_idx] += xgb.predict_proba(X_te)[:, 1] / CFG['n_folds']

        # Evaluate only on ORIGINAL validation samples for clean metric
        va_orig_mask = is_original[va_idx]
        if np.sum(va_orig_mask) > 0:
            va_probs_orig = oof_xgb_s2[va_idx, seed_idx][va_orig_mask]
            va_y_orig = y_va_fold[va_orig_mask]
            f1_orig = f1_score(va_y_orig, (va_probs_orig >= OPTIMAL_THRESHOLD).astype(int))
            fold_scores_s2['xgb'][seed].append(f1_orig)
        else:
            fold_scores_s2['xgb'][seed].append(0.0)
            f1_orig = 0.0

        dt = time.time() - t0
        print(f"    [XGBoost seed={seed}] Orig F1@{OPTIMAL_THRESHOLD:.3f}={f1_orig:.5f}  [{dt:.0f}s]")

        del xgb; gc.collect()

        # ================================================================
        # CATBOOST
        # ================================================================
        t0 = time.time()
        cb_params = CFG['catboost_params'].copy()
        cb_params['random_seed'] = seed

        cb = CatBoostClassifier(**cb_params, cat_features=cb_cat_indices)
        cb.fit(cb_tr_fold, y_tr_fold,
               eval_set=(cb_va_fold, y_va_fold),
               verbose=False)

        oof_cb_s2[va_idx, seed_idx] = cb.predict_proba(cb_va_fold)[:, 1]
        test_cb_s2[:, seed_idx] += cb.predict_proba(cb_test)[:, 1] / CFG['n_folds']

        if np.sum(va_orig_mask) > 0:
            va_probs_orig_cb = oof_cb_s2[va_idx, seed_idx][va_orig_mask]
            f1_orig_cb = f1_score(va_y_orig, (va_probs_orig_cb >= OPTIMAL_THRESHOLD).astype(int))
            fold_scores_s2['cb'][seed].append(f1_orig_cb)
        else:
            fold_scores_s2['cb'][seed].append(0.0)
            f1_orig_cb = 0.0

        dt_cb = time.time() - t0
        print(f"    [CatBoost seed={seed}] Orig F1@{OPTIMAL_THRESHOLD:.3f}={f1_orig_cb:.5f}  [{dt_cb:.0f}s]")

        del cb; gc.collect()

        # ================================================================
        # LIGHTGBM
        # ================================================================
        t0 = time.time()
        lgb_params = CFG['lgb_params'].copy()
        lgb_params['random_state'] = seed

        lgb = LGBMClassifier(**lgb_params)
        lgb.fit(X_tr_fold, y_tr_fold,
                eval_set=[(X_va_fold, y_va_fold)],
                eval_metric='logloss',
                callbacks=[early_stopping(200), log_evaluation(0)])

        oof_lgb_s2[va_idx, seed_idx] = lgb.predict_proba(X_va_fold)[:, 1]
        test_lgb_s2[:, seed_idx] += lgb.predict_proba(X_te)[:, 1] / CFG['n_folds']

        if np.sum(va_orig_mask) > 0:
            va_probs_orig_lgb = oof_lgb_s2[va_idx, seed_idx][va_orig_mask]
            f1_orig_lgb = f1_score(va_y_orig, (va_probs_orig_lgb >= OPTIMAL_THRESHOLD).astype(int))
            fold_scores_s2['lgb'][seed].append(f1_orig_lgb)
        else:
            fold_scores_s2['lgb'][seed].append(0.0)
            f1_orig_lgb = 0.0

        dt_lgb = time.time() - t0
        pos_rate_lgb = np.mean(oof_lgb_s2[va_idx, seed_idx] >= OPTIMAL_THRESHOLD) * 100
        print(f"    [LightGBM seed={seed}] Orig F1@{OPTIMAL_THRESHOLD:.3f}={f1_orig_lgb:.5f}  "
              f"pos@{OPTIMAL_THRESHOLD:.3f}={pos_rate_lgb:.1f}%  [{dt_lgb:.0f}s]")

        del lgb; gc.collect()

        t_fold_total = time.time() - t_fold_start
        print(f"    Fold total: {t_fold_total:.0f}s")

elapsed_s2 = time.time() - T_START
print(f"\n{'='*60}")
print(f"  STAGE 2 COMPLETE")
print(f"  Total elapsed: {elapsed_s2/60:.1f} min ({elapsed_s2:.0f}s)")
print(f"{'='*60}")



  CELL 10: STAGE 2 — RETRAIN ON AUGMENTED DATA

  ENSEMBLE SEED 42 (1/3) — AUGMENTED DATA

  --- Fold 1/5 ---
    Train: 75,858 rows (60,910 orig + 14,948 pseudo), pos=2,892 (3.81%)
    [XGBoost seed=42] Orig F1@0.140=0.29975  [17s]
    [CatBoost seed=42] Orig F1@0.140=0.32164  [210s]
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[706]	valid_0's binary_logloss: 0.112403
    [LightGBM seed=42] Orig F1@0.140=0.29386  pos@0.140=5.8%  [16s]
    Fold total: 243s

  --- Fold 2/5 ---
    Train: 75,858 rows (60,827 orig + 15,031 pseudo), pos=2,892 (3.81%)
    [XGBoost seed=42] Orig F1@0.140=0.29218  [12s]
    [CatBoost seed=42] Orig F1@0.140=0.30931  [235s]
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[645]	valid_0's binary_logloss: 0.112893
    [LightGBM seed=42] Orig F1@0.140=0.29625  pos@0.140=5.5%  [15s]
    Fold total: 262s

  --- Fold 3/5 ---
    Train: 75,858 rows (60,712 orig + 15,14

# CELL 11: Final OOF Analysis & Threshold Refinement

Evaluate Stage 2 OOF predictions on ORIGINAL training samples only.
Re-optimize threshold since pseudo-labeling may have shifted probability distributions.


In [12]:
# ===================================================================
# CELL 11: Final OOF Analysis & Threshold Refinement
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 11: FINAL OOF ANALYSIS & THRESHOLD REFINEMENT")
print("=" * 70)

# Extract OOF for ORIGINAL training samples only (unbiased metric)
oof_xgb_orig = np.nan_to_num(np.mean(oof_xgb_s2[is_original], axis=1), 0)
oof_cb_orig  = np.nan_to_num(np.mean(oof_cb_s2[is_original], axis=1), 0)
oof_lgb_orig = np.nan_to_num(np.mean(oof_lgb_s2[is_original], axis=1), 0)

# Uniform ensemble (all seeds, all models)
all_oof_s2 = np.hstack([oof_xgb_s2[is_original], oof_cb_s2[is_original], oof_lgb_s2[is_original]])
oof_uniform_s2 = np.nan_to_num(np.mean(all_oof_s2, axis=1), 0)

# Model-level average ensemble
oof_avg_s2 = (oof_xgb_orig + oof_cb_orig + oof_lgb_orig) / 3.0

print(f"\n  Stage 2 OOF (original samples only) probability stats:")
for label, arr in [('XGBoost', oof_xgb_orig), ('CatBoost', oof_cb_orig),
                    ('LightGBM', oof_lgb_orig), ('Uniform Ensemble', oof_uniform_s2)]:
    print(f"    {label:20s}: mean={np.mean(arr):.5f}  med={np.median(arr):.5f}  "
          f"p95={np.percentile(arr, 95):.4f}  p99={np.percentile(arr, 99):.4f}")

# --- Re-optimize threshold on Stage 2 OOF ---
print(f"\n  Threshold optimization on Stage 2 OOF (original samples only):")
print(f"  {'Model':20s} {'S1 Best T':>10s} {'S2 Best T':>10s} {'S2 F1':>10s} {'S2 P':>10s} {'S2 R':>10s}")
print(f"  {'-'*70}")

final_results = {}
best_overall_f1 = 0
best_overall_label = ''
best_overall_threshold = OPTIMAL_THRESHOLD
best_overall_oof = oof_uniform_s2

for label, oof_arr in [
    ('XGBoost', oof_xgb_orig),
    ('CatBoost', oof_cb_orig),
    ('LightGBM', oof_lgb_orig),
    ('Uniform Ensemble', oof_uniform_s2),
    ('Avg Ensemble', oof_avg_s2),
]:
    best_f1 = 0
    best_t = 0.5
    best_prec = 0
    best_rec = 0

    for t in thresholds:
        binary = (oof_arr >= t).astype(int)
        if np.sum(binary) == 0:
            continue
        f1 = f1_score(y, binary)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            best_prec = precision_score(y, binary, zero_division=0)
            best_rec = recall_score(y, binary, zero_division=0)

    s1_t = best_results.get(label, {}).get('threshold', 0.5)
    final_results[label] = {'threshold': best_t, 'f1': best_f1,
                             'precision': best_prec, 'recall': best_rec}
    print(f"  {label:20s} {s1_t:10.4f} {best_t:10.4f} {best_f1:10.5f} "
          f"{best_prec:10.4f} {best_rec:10.4f}")

    if best_f1 > best_overall_f1:
        best_overall_f1 = best_f1
        best_overall_label = label
        best_overall_threshold = best_t
        best_overall_oof = oof_arr

# --- Final Selection ---
FINAL_THRESHOLD = best_overall_threshold
BEST_OOF_F1_S2 = best_overall_f1

print(f"\n  ▶ FINAL SELECTION: {best_overall_label}")
print(f"    Optimal threshold: {FINAL_THRESHOLD:.4f}")
print(f"    OOF F1 (original samples): {BEST_OOF_F1_S2:.5f}")
print(f"    Stage 1 → Stage 2 delta: {BEST_OOF_F1_S2 - BEST_OOF_F1_S1:+.5f}")

# Stage 1 vs Stage 2 comparison
print(f"\n  Stage 1 vs Stage 2 comparison:")
print(f"    Stage 1 (no pseudo-labels): F1={BEST_OOF_F1_S1:.5f} @ threshold={OPTIMAL_THRESHOLD:.4f}")
print(f"    Stage 2 (with pseudo-labels): F1={BEST_OOF_F1_S2:.5f} @ threshold={FINAL_THRESHOLD:.4f}")
if BEST_OOF_F1_S2 > BEST_OOF_F1_S1:
    print(f"    ✅ Pseudo-labeling IMPROVED OOF by {BEST_OOF_F1_S2 - BEST_OOF_F1_S1:+.5f}")
else:
    print(f"    ⚠️  Pseudo-labeling did not improve OOF. Using Stage 1 predictions instead.")
    # Fall back to Stage 1
    FINAL_THRESHOLD = OPTIMAL_THRESHOLD
    BEST_OOF_F1_S2 = BEST_OOF_F1_S1

dbg.log_step("Final OOF analysis done",
             extra=f"threshold={FINAL_THRESHOLD:.4f}, OOF F1={BEST_OOF_F1_S2:.5f}")



  CELL 11: FINAL OOF ANALYSIS & THRESHOLD REFINEMENT

  Stage 2 OOF (original samples only) probability stats:
    XGBoost             : mean=0.04094  med=0.01424  p95=0.1662  p99=0.4487
    CatBoost            : mean=0.03982  med=0.01418  p95=0.1598  p99=0.4368
    LightGBM            : mean=0.04067  med=0.01455  p95=0.1638  p99=0.4377
    Uniform Ensemble    : mean=0.04048  med=0.01450  p95=0.1622  p99=0.4393

  Threshold optimization on Stage 2 OOF (original samples only):
  Model                 S1 Best T  S2 Best T      S2 F1       S2 P       S2 R
  ----------------------------------------------------------------------
  XGBoost                  0.1300     0.1600    0.29085     0.2546     0.3391
  CatBoost                 0.1400     0.1500    0.30579     0.2645     0.3624
  LightGBM                 0.1400     0.1650    0.29146     0.2624     0.3278
  Uniform Ensemble         0.1400     0.1600    0.29819     0.2653     0.3404
  Avg Ensemble             0.5000     0.1600    0.29819

# CELL 12: Final Test Predictions & Binary Submission

Convert test probabilities to BINARY (0/1) using the optimal threshold found on OOF data.
This binary submission is what Kaggle evaluates at their strict 0.5 threshold.
Since we already converted to 0/1 at the optimal cutoff, Kaggle's 0.5 is effectively
replaced by our optimal threshold.


In [13]:
# ===================================================================
# CELL 12: Final Test Predictions & Binary Submission
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 12: FINAL TEST PREDICTIONS & BINARY SUBMISSION")
print("=" * 70)

# Average test predictions across seeds (Stage 2)
test_xgb_avg = np.mean(test_xgb_s2, axis=1)
test_cb_avg  = np.mean(test_cb_s2, axis=1)
test_lgb_avg = np.mean(test_lgb_s2, axis=1)

# Multiple ensemble strategies for comparison
test_uniform = np.mean(np.hstack([test_xgb_s2, test_cb_s2, test_lgb_s2]), axis=1)
test_avg = (test_xgb_avg + test_cb_avg + test_lgb_avg) / 3.0

# Use the best ensemble from OOF analysis
if 'Uniform' in best_overall_label:
    final_probs = test_uniform
elif 'Avg' in best_overall_label:
    final_probs = test_avg
elif 'XGBoost' in best_overall_label:
    final_probs = test_xgb_avg
elif 'CatBoost' in best_overall_label:
    final_probs = test_cb_avg
elif 'LightGBM' in best_overall_label:
    final_probs = test_lgb_avg
else:
    final_probs = test_uniform

print(f"\n  Final ensemble: {best_overall_label}")
print(f"  Optimal threshold: {FINAL_THRESHOLD:.4f}")

# Print test prediction stats
print(f"\n  Test prediction statistics (Stage 2):")
for label, probs in [
    ('XGBoost', test_xgb_avg),
    ('CatBoost', test_cb_avg),
    ('LightGBM', test_lgb_avg),
    ('Uniform Ensemble', test_uniform),
    ('Avg Ensemble', test_avg),
]:
    pos_at_t = np.mean(probs >= FINAL_THRESHOLD) * 100
    print(f"  {label:25s}: mean={np.mean(probs):.5f}  med={np.median(probs):.5f}  "
          f"pos@{FINAL_THRESHOLD:.3f}={pos_at_t:.2f}%  "
          f"range=[{probs.min():.4f}, {probs.max():.4f}]")

# --- BINARY CONVERSION ---
# Apply optimal threshold to convert probabilities to 0/1
binary_predictions = (final_probs >= FINAL_THRESHOLD).astype(int)

n_pos_pred = np.sum(binary_predictions)
n_neg_pred = np.sum(binary_predictions == 0)
pos_rate_pred = n_pos_pred / len(binary_predictions) * 100

print(f"\n  Binary prediction stats:")
print(f"    Class 0 predicted: {n_neg_pred:,} ({100*n_neg_pred/len(binary_predictions):.2f}%)")
print(f"    Class 1 predicted: {n_pos_pred:,} ({pos_rate_pred:.2f}%)")
print(f"    Threshold used:    {FINAL_THRESHOLD:.4f}")

# Sanity checks
if pos_rate_pred < 0.5:
    print(f"    ⚠️  WARNING: Very low positive rate ({pos_rate_pred:.2f}%)")
if pos_rate_pred > 50:
    print(f"    ⚠️  WARNING: Very high positive rate ({pos_rate_pred:.2f}%)")
if 1.0 < pos_rate_pred < 8.0:
    print(f"    ✅ Positive rate in reasonable range (1-8%)")

# --- Create Submission ---
submission = sub_raw.copy()
submission['TARGET'] = binary_predictions

# Save
output_dir = Path('/kaggle/working')
output_dir.mkdir(exist_ok=True)

sub_path = output_dir / 'submission_binary_v6.csv'
submission.to_csv(sub_path, index=False)
print(f"\n  ✓ Binary submission saved to: {sub_path}")
print(f"    Shape: {submission.shape}")
print(f"    First 10 rows:")
print(submission.head(10).to_string())
print(f"\n    Value counts:")
print(f"    {submission['TARGET'].value_counts().to_string()}")

# Show probability distribution for reference
print(f"\n  Probability distribution of submitted predictions:")
print(f"    Mean:  {np.mean(final_probs):.5f}")
print(f"    Median: {np.median(final_probs):.5f}")
print(f"    Std:   {np.std(final_probs):.5f}")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"    P{p:2d}:   {np.percentile(final_probs, p):.5f}")

dbg.log_step("Submission saved", shape=submission.shape,
             extra=f"binary, pos_rate={pos_rate_pred:.2f}%, threshold={FINAL_THRESHOLD:.4f}")



  CELL 12: FINAL TEST PREDICTIONS & BINARY SUBMISSION

  Final ensemble: CatBoost
  Optimal threshold: 0.1500

  Test prediction statistics (Stage 2):
  XGBoost                  : mean=0.03915  med=0.01443  pos@0.150=5.10%  range=[0.0005, 0.9399]
  CatBoost                 : mean=0.03811  med=0.01431  pos@0.150=4.81%  range=[0.0003, 0.9821]
  LightGBM                 : mean=0.03903  med=0.01468  pos@0.150=4.99%  range=[0.0003, 0.9214]
  Uniform Ensemble         : mean=0.03876  med=0.01461  pos@0.150=4.92%  range=[0.0004, 0.9478]
  Avg Ensemble             : mean=0.03876  med=0.01461  pos@0.150=4.92%  range=[0.0004, 0.9478]

  Binary prediction stats:
    Class 0 predicted: 57,736 (95.19%)
    Class 1 predicted: 2,918 (4.81%)
    Threshold used:    0.1500
    ✅ Positive rate in reasonable range (1-8%)

  ✓ Binary submission saved to: /kaggle/working/submission_binary_v6.csv
    Shape: (60654, 2)
    First 10 rows:
      id  TARGET
0   3496       0
1  17271       0
2  44259       0
3  6

# CELL 13: V6 Performance Dashboard


In [14]:
# ===================================================================
# CELL 13: V6 PERFORMANCE DASHBOARD
# ===================================================================
print("\n" + "=" * 70)
print("  CELL 13: V6 PERFORMANCE DASHBOARD")
print("=" * 70)

# --- Model Health ---
print(f"\n  {'='*60}")
print(f"  MODEL HEALTH CHECK")
print(f"  {'='*60}")

for model_name, oof_s1, oof_s2 in [
    ('XGBoost', oof_xgb_s1, oof_xgb_s2),
    ('CatBoost', oof_cb_s1, oof_cb_s2),
    ('LightGBM', oof_lgb_s1, oof_lgb_s2),
]:
    # S1
    s1_avg = np.nan_to_num(np.mean(oof_s1, axis=1), 0)
    s1_f1 = f1_score(y, (s1_avg >= OPTIMAL_THRESHOLD).astype(int))
    s1_mean = np.mean(s1_avg)

    # S2 (original samples)
    s2_avg = np.nan_to_num(np.mean(oof_s2[is_original], axis=1), 0)
    s2_f1 = f1_score(y, (s2_avg >= FINAL_THRESHOLD).astype(int))
    s2_mean = np.mean(s2_avg)

    status = "✅" if s2_f1 > 0.05 else "🔴 DEAD"
    print(f"  {status} {model_name:12s}: S1 mean_prob={s1_mean:.4f} F1@{OPTIMAL_THRESHOLD:.3f}={s1_f1:.5f}  "
          f"→ S2 mean_prob={s2_mean:.4f} F1@{FINAL_THRESHOLD:.3f}={s2_f1:.5f}")

# --- Feature Summary ---
print(f"\n  {'='*60}")
print(f"  FEATURE SUMMARY")
print(f"  {'='*60}")
print(f"  Numerical (after adv filter):  {n_kept}")
print(f"  PCA components (ddof=0):       {n_pca}")
print(f"  Row statistics:                {n_rs}")
print(f"  KMeans clusters:               {n_km}")
print(f"  Label-encoded cats:            {n_cat}")
print(f"  {'─' * 40}")
print(f"  TOTAL FEATURES:                {n_total}")
print(f"  Features dropped (adv):        {len(drift_features)}")
print(f"  CatBoost cat_features:         {len(cb_cat_indices)}")

# --- Version Comparison ---
print(f"\n  {'='*60}")
print(f"  VERSION COMPARISON (All)")
print(f"  {'='*60}")
print(f"  {'Version':12s} {'LB F1':>10s} {'OOF F1':>10s} {'CV→LB Gap':>12s} {'Runtime':>8s} {'Strategy':>20s}")
print(f"  {'-'*70}")
print(f"  {'V1':12s} {'0.19568':>10s} {'0.3624':>10s} {'46.0%':>12s} {'~2h':>8s} {'SMOTE+Weights':>20s}")
print(f"  {'V2':12s} {'0.22576':>10s} {'0.2872':>10s} {'21.4%':>12s} {'~1h':>8s} {'CB-only+SMOTE':>20s}")
print(f"  {'V3':12s} {'0.19233':>10s} {'0.2910':>10s} {'34.0%':>12s} {'~3h':>8s} {'SMOTE+3models':>20s}")
print(f"  {'V4':12s} {'CANCELLED':>10s} {'N/A':>10s} {'N/A':>12s} {'11.5h':>8s} {'261cats☠️':>20s}")
print(f"  {'V5':12s} {'---':>10s} {'0.0675':>10s} {'---':>12s} {'0.5h':>8s} {'SMOTE kills all':>20s}")
print(f"  {'V6':12s} {'---':>10s} {f'{BEST_OOF_F1_S2:.4f}':>10s} {'---':>12s} "
      f"{f'{elapsed_s2/60:.0f}min':>8s} {'PureLogLoss+Adv+PCA':>20s}")
print(f"  {'V6 S1 only':12s} {'---':>10s} {f'{BEST_OOF_F1_S1:.4f}':>10s} {'---':>12s} "
      f"{f'{elapsed_s1/60:.0f}min':>8s} {'Before pseudo-label':>20s}")

# --- Key Metrics ---
print(f"\n  {'='*60}")
print(f"  KEY METRICS")
print(f"  {'='*60}")
print(f"  Adversarial ROC-AUC:           {adv_roc:.4f}")
print(f"  Features dropped (drift):      {len(drift_features)}/{len(numerical_features)}")
print(f"  PCA components:                {n_pca} (ddof=0)")
print(f"  Stage 1 optimal threshold:     {OPTIMAL_THRESHOLD:.4f}")
print(f"  Stage 1 best OOF F1:           {BEST_OOF_F1_S1:.5f}")
print(f"  Pseudo-labels added:           {n_pseudo_pos} pos + {n_pseudo_neg} neg")
print(f"  Stage 2 final threshold:       {FINAL_THRESHOLD:.4f}")
print(f"  Stage 2 best OOF F1:           {BEST_OOF_F1_S2:.5f}")
print(f"  Test predicted pos rate:       {pos_rate_pred:.2f}%")
print(f"  Test predicted pos count:      {n_pos_pred:,}")

# --- Target ---
print(f"\n  {'='*60}")
print(f"  TARGET vs TOP TEAMS")
print(f"  {'='*60}")
print(f"  Top team LB F1:       0.30103+")
print(f"  V2 best LB F1:        0.22576")
print(f"  V6 OOF F1 (S2):       {BEST_OOF_F1_S2:.5f}")
print(f"  V6 target LB:         0.28–0.32")

# --- Debug Summary ---
dbg.summary()

total_time = time.time() - T_START
print(f"\n  {'='*60}")
print(f"  V6 COMPLETE ✅")
print(f"  Total runtime: {total_time/60:.1f} min ({total_time:.0f}s)")
print(f"  Submit submission_binary_v6.csv to Kaggle")
print(f"  Optimal threshold used: {FINAL_THRESHOLD:.4f}")
print(f"  {'='*60}")



  CELL 13: V6 PERFORMANCE DASHBOARD

  MODEL HEALTH CHECK
  ✅ XGBoost     : S1 mean_prob=0.0383 F1@0.140=0.28699  → S2 mean_prob=0.0409 F1@0.150=0.28797
  ✅ CatBoost    : S1 mean_prob=0.0373 F1@0.140=0.30150  → S2 mean_prob=0.0398 F1@0.150=0.30579
  ✅ LightGBM    : S1 mean_prob=0.0380 F1@0.140=0.29010  → S2 mean_prob=0.0407 F1@0.150=0.28959

  FEATURE SUMMARY
  Numerical (after adv filter):  327
  PCA components (ddof=0):       47
  Row statistics:                11
  KMeans clusters:               2
  Label-encoded cats:            6
  ────────────────────────────────────────
  TOTAL FEATURES:                66
  Features dropped (adv):        17
  CatBoost cat_features:         8

  VERSION COMPARISON (All)
  Version           LB F1     OOF F1    CV→LB Gap  Runtime             Strategy
  ----------------------------------------------------------------------
  V1              0.19568     0.3624        46.0%      ~2h        SMOTE+Weights
  V2              0.22576     0.2872        21.